## Logistic Regression Project
### Enhanced Breast Cancer CSV Dataset

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

## 1) Cleaning Dataset

In [2]:
df = pd.read_csv('../dataset/breast_cancer_enhanced_dataset.csv')
df.head(20)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,shape_irregularity,border_complexity,tumor_aggressiveness,radius_texture_interaction,radius_concavity_interaction,concavity_density,malignancy_risk_score
0,5.683796e+06,B,11.829858,21.726166,75.154378,435.022394,0.087089,0.050717,0.01586811960710508,0.011641,0.078226,0.000185,0.026075,257.017451,0.187718,0.000036,22.433798
1,-6.253379e+06,B,10.991150,17.103260,71.798929,381.386295,0.089339,0.109498,0.09734386385684672,0.035629,0.242470,0.003468,0.080823,187.984489,1.069921,0.000255,24.021839
2,4.213892e+06,M,21.433519,15.092437,142.753006,1392.399890,0.099557,0.152079,0.1933722235754475,0.126922,0.472373,0.024543,0.157458,323.484034,4.144647,0.000139,49.053992
3,-2.986069e+06,B,11.700452,14.872127,74.154481,404.112556,0.101291,0.077563,0.043749161495531624,0.028747,0.150059,0.001258,0.050020,174.010599,0.511885,0.000108,23.276145
4,6.469594e+05,B,13.259377,17.212990,83.621014,521.124238,0.072905,0.043312,0.04698257853597487,0.010096,0.100391,0.000474,0.033464,228.233523,0.622960,0.000090,25.647074
5,-4.581518e+05,B,11.620937,16.155888,76.026791,436.142831,0.112669,0.095446,0.03670357148832927,0.031314,0.163464,0.001149,0.054488,187.746558,0.426530,0.000084,23.612382
6,6.104780e+06,B,12.166927,14.868031,78.601322,428.659208,0.103159,0.090379,0.0667781997494826,0.027997,0.185154,0.001870,0.061718,180.898245,0.812485,0.000156,25.061439
7,9.398100e+07,B,13.443279,18.669093,87.810591,564.287299,0.106001,0.115531,0.08396727311815218,0.054183,0.253681,0.004550,0.084560,250.973833,1.128795,0.000149,28.767221
8,2.722075e+06,B,8.743292,15.826895,54.991483,253.178665,0.114348,0.080234,0.04082440590664949,0.019867,0.140925,0.000811,0.046975,138.379160,0.356940,0.000161,17.450987
9,1.403596e+06,M,18.083293,18.692756,120.690753,1034.577981,0.115221,0.150960,0.17747014589097307,0.105356,0.433786,0.018698,0.144595,338.026597,3.209245,0.000172,41.764167


#### id column is not required since it has no correlation with the target variable

In [4]:
df1 = df.drop(['id', 'malignancy_risk_score', 'shape_irregularity', 'border_complexity', 'tumor_aggressiveness', 'radius_texture_interaction', 'radius_concavity_interaction', 'concavity_density'], axis = 'columns')
df1.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean
0,B,11.829858,21.726166,75.154378,435.022394,0.087089,0.050717,0.01586811960710508,0.011641
1,B,10.991150,17.103260,71.798929,381.386295,0.089339,0.109498,0.09734386385684672,0.035629
2,M,21.433519,15.092437,142.753006,1392.399890,0.099557,0.152079,0.1933722235754475,0.126922
3,B,11.700452,14.872127,74.154481,404.112556,0.101291,0.077563,0.043749161495531624,0.028747
4,B,13.259377,17.212990,83.621014,521.124238,0.072905,0.043312,0.04698257853597487,0.010096


In [6]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 5500 entries, 0 to 5499
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   diagnosis            5500 non-null   str    
 1   radius_mean          5500 non-null   float64
 2   texture_mean         5500 non-null   float64
 3   perimeter_mean       5500 non-null   float64
 4   area_mean            5500 non-null   float64
 5   smoothness_mean      5500 non-null   float64
 6   compactness_mean     5500 non-null   float64
 7   concavity_mean       5500 non-null   str    
 8   concave points_mean  5500 non-null   float64
dtypes: float64(7), str(2)
memory usage: 386.8 KB


#### Great that there are no empty (NaN) fields, we do not have to drop any rows
#### We have to do data comversions:
- concavity_mean --> Str to float64
- diagnosis (Target Variable) ---> Use simple binary classification (0: Benign & 1: Malignant)

#### Convert Datatype for concavity_mean (String to Float64)

In [8]:
df1['concavity_mean'] = df1['concavity_mean'].apply(lambda x: x[:12])
df1['concavity_mean'] = df1['concavity_mean'].apply(lambda x: float(x))
df1['concavity_mean']

0       0.015868
1       0.097344
2       0.193372
3       0.043749
4       0.046983
          ...   
5495    0.069515
5496    0.012322
5497    0.164108
5498    0.316075
5499    0.146180
Name: concavity_mean, Length: 5500, dtype: float64

#### Use lambda function (0: Benign & 1: Malignant)

In [9]:
df1['diagnosis'] = df1['diagnosis'].map({'M': 1, 'B':0})
df1.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean
0,0,11.829858,21.726166,75.154378,435.022394,0.087089,0.050717,0.015868,0.011641
1,0,10.991150,17.103260,71.798929,381.386295,0.089339,0.109498,0.097344,0.035629
2,1,21.433519,15.092437,142.753006,1392.399890,0.099557,0.152079,0.193372,0.126922
3,0,11.700452,14.872127,74.154481,404.112556,0.101291,0.077563,0.043749,0.028747
4,0,13.259377,17.212990,83.621014,521.124238,0.072905,0.043312,0.046983,0.010096


In [10]:
df1.describe()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean
count,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000
mean,0.369091,14.095290,19.317068,91.749574,651.332914,0.096027,0.103069,0.083756,0.048172
std,0.482602,3.484868,4.350629,24.012506,346.027394,0.013878,0.052939,0.246983,0.038531
min,0.000000,6.817646,9.522736,42.639188,123.928240,0.052200,0.015551,-9.269280,-0.002668
25%,0.000000,11.679531,16.196326,75.119335,420.093960,0.085863,0.063526,0.028792,0.019967
50%,0.000000,13.339209,18.817023,86.222090,549.472961,0.095362,0.090793,0.058966,0.032845
75%,1.000000,15.760529,21.841027,103.688058,776.638240,0.105008,0.129659,0.126078,0.072017
max,1.000000,28.341004,39.442971,188.983008,2507.982355,0.164055,0.348835,7.607783,0.202662


## 2) Data Splitting into Train and Test Set

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
features = df1.drop(['diagnosis'], axis='columns')
target = df1['diagnosis']

In [13]:
features

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean
0,11.829858,21.726166,75.154378,435.022394,0.087089,0.050717,0.015868,0.011641
1,10.991150,17.103260,71.798929,381.386295,0.089339,0.109498,0.097344,0.035629
2,21.433519,15.092437,142.753006,1392.399890,0.099557,0.152079,0.193372,0.126922
3,11.700452,14.872127,74.154481,404.112556,0.101291,0.077563,0.043749,0.028747
4,13.259377,17.212990,83.621014,521.124238,0.072905,0.043312,0.046983,0.010096
...,...,...,...,...,...,...,...,...
5495,11.688492,18.217548,75.277156,429.895271,0.113777,0.104274,0.069515,0.034620
5496,13.831066,19.454900,88.204429,580.072530,0.087240,0.061584,0.012322,0.022518
5497,19.403923,18.152520,127.631265,1129.627896,0.104142,0.144262,0.164108,0.094511
5498,20.946319,25.233666,142.498720,1340.103257,0.109454,0.226277,0.316075,0.148336


In [14]:
target

0       0
1       0
2       1
3       0
4       0
       ..
5495    0
5496    0
5497    1
5498    1
5499    1
Name: diagnosis, Length: 5500, dtype: int64

In [15]:
x_train, x_test, y_train, y_test = train_test_split(features, target, train_size=0.8, random_state=1)
print(f'x_train shape: {x_train.shape} | x_test dimensions: {x_train.ndim}')
print(f'x_test shape: {x_test.shape} | x_test dimensions: {x_test.ndim}')
print(f'y_train shape: {y_train.shape} | y_train dimensions: {y_train.ndim}')
print(f'y_test shape: {y_test.shape} | y_test dimensions: {y_test.ndim}')

x_train shape: (4400, 8) | x_test dimensions: 2
x_test shape: (1100, 8) | x_test dimensions: 2
y_train shape: (4400,) | y_train dimensions: 1
y_test shape: (1100,) | y_test dimensions: 1


#### Since there are different in ranges in values for all the features, to ensure that model is able to make patterns through learning based on features of the same range
#### we will use standard scalar on both x_train and x_test

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

#### Before scaling using StandardScaler

In [17]:
x_train.head()

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean
3332,16.108940,18.139588,106.166647,808.426533,0.096408,0.112682,0.097807,0.059905
2161,14.965157,17.654707,96.805536,674.077664,0.116350,0.130791,0.153983,0.086574
3192,10.949432,14.988412,70.847732,364.580943,0.080321,0.071271,0.039757,0.022050
2708,11.826760,21.310931,76.372018,441.042877,0.097915,0.078490,0.025907,0.021163
2139,14.962159,15.518628,97.457922,688.159160,0.084043,0.110001,0.064256,0.036750


In [18]:
x_test.head()

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean
4410,9.005977,17.197827,58.696874,256.634741,0.106350,0.140737,0.309322,0.043847
2256,14.446327,13.944902,96.054030,646.698644,0.109934,0.093691,0.068427,0.064049
3684,10.464182,27.612122,64.778754,315.611996,0.089831,0.076417,0.061925,0.028713
4706,13.531830,22.031813,88.096194,567.568480,0.079745,0.082801,0.042175,0.020380
5429,16.637001,20.211768,107.141154,865.662830,0.075429,0.070934,0.037429,0.022376


#### After scaling using Standard Scaler

In [19]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [20]:
df_scaled_x_train = pd.DataFrame(x_train_scaled, columns = x_train.columns)
print(df_scaled_x_train.head())
print(df_scaled_x_train.shape)

   radius_mean  texture_mean  perimeter_mean  area_mean  smoothness_mean  \
0     0.564229     -0.263159        0.587329   0.439498         0.021749   
1     0.238726     -0.375047        0.200501   0.055406         1.455267   
2    -0.904089     -0.990307       -0.872150  -0.829418        -1.134557   
3    -0.654415      0.468642       -0.643870  -0.610820         0.130091   
4     0.237873     -0.867957        0.227460   0.095664        -0.867041   

   compactness_mean  concavity_mean  concave points_mean  
0          0.177858        0.051966             0.292396  
1          0.521378        0.257563             0.980603  
2         -0.607656       -0.160489            -0.684485  
3         -0.470719       -0.211176            -0.707392  
4          0.127003       -0.070825            -0.305152  
(4400, 8)


In [21]:
df_scaled_x_test = pd.DataFrame(x_test_scaled, columns = x_test.columns)
print(df_scaled_x_test.head())
print(df_scaled_x_test.shape)

   radius_mean  texture_mean  perimeter_mean  area_mean  smoothness_mean  \
0    -1.457167     -0.480474       -1.374258  -1.138028         0.736438   
1     0.091075     -1.231101        0.169447  -0.022868         0.994050   
2    -1.042184      1.922670       -1.122937  -0.969417        -0.450995   
3    -0.169177      0.634989       -0.159394  -0.249094        -1.175945   
4     0.714508      0.215006        0.627599   0.603132        -1.486240   

   compactness_mean  concavity_mean  concave points_mean  
0          0.710027        0.826083            -0.121990  
1         -0.182385       -0.055559             0.399323  
2         -0.510042       -0.079357            -0.512548  
3         -0.388957       -0.151638            -0.727595  
4         -0.614052       -0.169007            -0.676085  
(1100, 8)


#### Now with all values in all features being in the same ranges, comparison can be made better. Proceed with training of model

In [22]:
logistic_regression_model = LogisticRegression()
logistic_regression_model.fit(x_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [23]:
print(f'Coefficients: {logistic_regression_model.coef_}')

Coefficients: [[ 0.12883647  1.3681902  -0.06814744  3.10838178  0.69019948 -0.55427285
   0.26452076  3.28707732]]


In [24]:
print(f'Intercept: {logistic_regression_model.intercept_}')

Intercept: [-0.37905365]


## 3) Evaluation of Model's accuracy/performance using Test Dataset (check generalisability)

In [25]:
predictions = logistic_regression_model.predict(x_test_scaled)
predictions

array([0, 0, 0, ..., 1, 1, 0], shape=(1100,))

### Side to side comparison between ground truth values and predicted values

In [26]:
results = pd.DataFrame({
    'Ground Truth': y_test.reset_index(drop=True),
    'Predictions': predictions
})

print(results.head(20))

    Ground Truth  Predictions
0              0            0
1              0            0
2              0            0
3              0            0
4              1            0
5              0            0
6              0            0
7              0            0
8              1            1
9              0            0
10             1            1
11             1            1
12             0            0
13             0            0
14             0            0
15             1            1
16             1            1
17             0            0
18             0            0
19             1            1


In [27]:
df1['diagnosis'].value_counts()

diagnosis
0    3470
1    2030
Name: count, dtype: int64

#### The dataset is moderately imbalanced, with benign cases occurring more frequently than malignant cases.
- There is 1440 more cases on breast cancer being benign than malignent

In [28]:
from sklearn.metrics import classification_report, confusion_matrix

### Confusion Matrix

In [29]:
cm = confusion_matrix(y_test, predictions)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:\n", cm)
print(f"TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")


Confusion Matrix:
 [[680  22]
 [ 49 349]]
TN: 680, FP: 22, FN: 49, TP: 349


#### Evaluation on the model's prediction
- TN (True Negative) --> Predicted Correctly (Predicted: Benign | Actual: Benign)
- FP (False Positive) --> Predicted Wrongly (Predicted: Malignant |  Actual: Benign)
- FN (False Negative) --> Predicted Wrongly (Predicted: Benign |  Actual: Malignant)
- TP (True Positive) --> Predicted Correctly (Predicted: Malignant |  Actual: Malignant)


### Most important (Serious issue that has to be corrected):
- FN
- It will be detrimental for patient to be deemed as cancer free (Benign) when he actually suffering from a malignant case of cancer --> Health Hazard

### Classification Report

In [30]:
cr = classification_report(y_test, predictions)
print(cr)

              precision    recall  f1-score   support

           0       0.93      0.97      0.95       702
           1       0.94      0.88      0.91       398

    accuracy                           0.94      1100
   macro avg       0.94      0.92      0.93      1100
weighted avg       0.94      0.94      0.93      1100



#### Evaluation on the model's prediction
- The model achieves a recall of 88% for malignant cases, meaning approximately 11% of actual malignant tumors are misclassified as benign. In a medical context, minimizing false negatives is especially important because missed cancer diagnoses may delay treatment and may be fatal in the worst case.